In [ ]:
# Import Library
import os
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
# Load Dataset
PATH = 'dataset/'
CLASS_NAMES = os.listdir(PATH)
print(CLASS_NAMES)

In [ ]:
# Load Image
rows, cols = 224, 224
x, y = [], []
counter = 0

for class_name in CLASS_NAMES:
    for img_name in os.listdir(PATH + class_name):
        path = f"{PATH}{class_name}/{img_name}"
        label = counter
        image = tf.keras.preprocessing.image.load_img(path, target_size=(rows, cols), color_mode="rgb")
        image = tf.keras.preprocessing.image.img_to_array(image)
        x.append(image)
        y.append(label)
    counter += 1
    print(class_name)
    
x = np.array(x)
y = np.array(y)

x = x.astype('float32') / 255.0

print(x.shape)
print(y.shape)
        

In [ ]:
# splitting
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_val, x_test, y_val , y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
# Preprocessing OHE
y_train = keras.utils.to_categorical(y_train, num_classes=4)
y_val = keras.utils.to_categorical(y_val, num_classes=4)
y_test = keras.utils.to_categorical(y_test, num_classes=4)

In [ ]:
# DCNN
model = keras.Sequential([
    keras.layers.Input(shape=(rows, cols, 3)),
    
    keras.layers.Conv2D(8, (5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),
    keras.layers.Conv2D(16, (5, 5), activation="relu"),
    keras.layers.Conv2D(16, (5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),
    keras.layers.Flatten(),
    
    keras.layers.Dense(120, activation="relu"),
    keras.layers.Dense(4, activation="softmax"),
])

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=32, epochs=10, validation_data=(x_val, y_val))

In [ ]:
# model evaluation
model.evaluate(x_test, y_test)

In [ ]:
pred_result = model.predict(x_test)

for i in range(3):
    pred = CLASS_NAMES[np.argmax(pred_result[i])]
    true = CLASS_NAMES[np.argmax(y_test[i])]
    
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title(f"pred: {pred} | True: {true}")
    plt.axis("off") 
    plt.show()

In [ ]:
# Plot evaluation metrics
plt.figure(figsize=(20,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('epochs')
plt.ylabel('acc')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()
plt.show()